# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup
First, let's print a simple message to ensure our environment is set up correctly.

In [1]:
print("Hello World")

Hello World


Then, let's add the directory containing src to path

In [2]:
import os
#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability


## 2. Checking System Memory
We will check the available system memory to ensure that we have enough resources to load and run the model. The following command outputs the total, free, and available memory in gigabytes.

In [5]:
#!pip show bitsandbytes
#!pip show acceleratec c -
#!pip show transformers
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

MemTotal: 251.77 GB
MemFree: 145.92 GB
MemAvailable: 186.54 GB
Free GPU Memory (GB): 0.00976562


## 3. Code Formatting and Linting
We use `black` for code formatting and `pylint` for linting to ensure our code is clean and follows best practices.


In [ ]:
!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

## 4. Loading Environment Variables
We load the Hugging Face token from an environment variable to authenticate our session. This token is necessary to access the model from the Hugging Face Hub.


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

## 5. Checking CUDA Availability
We check if CUDA is available on the system. CUDA is essential for running the model on GPU, which significantly speeds up the computations.


In [ ]:
import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

## 6.1. AutoModelForCausalLM Generation for TinyLlama

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

model_family, model_identifier = model_name.split("/")

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

Generating memory footprint of the model in Gigabytes

In [2]:
memory_footprint = model.get_memory_footprint()
print(f"Model Memory Footprint: {(memory_footprint / (1024 ** 3)):.2f} GB")

Model Memory Footprint: 4.10 GB


Example inference

In [ ]:
input_text = "What famous tower is in Paris?"
input_ids = tokenizer(input_text, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=input_ids["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Generated Text:", generated_text)

In [ ]:
import json

results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 7. Loading WikiText

In [4]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

directory_dataset = os.getcwd()
wikitext_batch_size = 1  # Just use batch size 1 for this project
#wikitext_batch_size = 16
#wikitext_batch_size = 64
sequence_length = 512  # Maximum sequence length - use the default value
wikitext_seed = 1

wikitext_data_module = WikiTextDataModule(
  directory_dataset=directory_dataset,
  batch_size=wikitext_batch_size,
  sequence_length=sequence_length,
  tokenizer_name=model_name,
  seed=wikitext_seed
)

#wikitext_train_dataloader = wikitext_data_module.train_dataloader()
wikitext_dataloader = wikitext_data_module.val_dataloader()

Token indices sequence length is longer than the specified maximum sequence length for this model (294896 > 2048). Running this sequence through the model will result in indexing errors


Printing contents of the dataset

In [5]:
print("Length of datasets:", len(wikitext_data_module.train_dataset), len(wikitext_data_module.val_dataset), len(wikitext_data_module.test_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

dataset_size = len(wikitext_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


Length of datasets: 36718 3760 4358
Number of batches in train_dataloader: 575
Batch 1:
  Original Text:   = Homarus gammarus = 
   Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , 
  Input data (first 5 tokens): tensor([    1,   259,   353, 15089, 26465])
  Target labels (first 5 tokens): tensor([  259,   353, 15089, 26465, 24988])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])
Batch 2:
  Original Text: , with spots that coalesce , and yellow below . The red colour associated with lobsters only appears after cooking . This occurs b

In [6]:
wikitext_dataset = []

# Loop through each batch in the dataloader
for batch in wikitext_dataloader:
  # Assuming the batch contains input_ids (tokenized text) and labels
  input_ids, labels = batch
  
  # Decode the input IDs back to text using the tokenizer
  decoded_text = wikitext_data_module.tokenizer.decode(input_ids[0].tolist())  # Assuming first element in batch
  wikitext_dataset.append(decoded_text)


In [7]:
print(f"Sample texts from train dataloader: {wikitext_dataset[1][:1000]}")
print(f"Type: {type(wikitext_dataset)}, Length: {len(wikitext_dataset)}")

Sample texts from train dataloader: , with spots that coalesce , and yellow below . The red colour associated with lobsters only appears after cooking . This occurs because , in life , the red pigment astaxanthin is bound to a protein complex , but the complex is broken up by the heat of cooking , releasing the red pigment . 
  The closest relative of H. gammarus is the American lobster , Homarus americanus . The two species are very similar , and can be crossed artificially , although hybrids are unlikely to occur in the wild since their ranges do not overlap . The two species can be distinguished by a number of characteristics : 
  The rostrum of H. americanus bears one or more spines on the underside , which are lacking in H. gammarus . 
  The spines on the claws of H. americanus are red or red @-@ tipped , while those of H. gammarus are white or white @-@ tipped . 
  The underside of the claw of H. americanus is orange or red , while that of H. gammarus is creamy white or very pale

## 8. Quantization

In [8]:
device="cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

### 8.1. BitsAndBytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    # llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)
bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    # bnb_4bit_quant_type="nf4"
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype=torch.float32,
    device_map=device
)

In [ ]:
print(f"8bit {model} Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")
print(f"4bit Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

### 7.2 AWQ

Hyperparameters and config

In [9]:
awq_calib_split="validation"

awq_model_path = f'{model_identifier}-awq'
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM" 
}

Quantization

In [12]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map=device
)

# Quantize with wikitext validation as calibration data
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=wikitext_dataset,  # Pass the loaded validation dataset here
)

# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
AWQ: 100%|██████████| 22/22 [02:55<00:00,  7.99s/it]


Model is quantized and saved at "TinyLlama-1.1B-Chat-v1.0-awq"


In [22]:
import os

def calculate_model_size(model_path):
  """
  This function calculates the total size of a model directory containing saved model files.

  Args:
      model_path (str): The path to the directory containing the model files.

  Returns:
      None: The function directly prints the total model size in human-readable format.
  """

  # Initialize total size variable
  total_size = 0

  # Loop through files in the model directory
  for filename in os.listdir(model_path):
    file_path = os.path.join(model_path, filename)
    # Check if it's a file (not a directory)
    if os.path.isfile(file_path):
      file_size = os.path.getsize(file_path)
      total_size += file_size

  # Convert to human-readable format
  if total_size > 1024**2:
    total_size_mb = total_size / (1024**2)
    print(f"Total Model Size for {model_path}: {total_size_mb:.2f} MB")
  elif total_size > 1024:
    total_size_kb = total_size / 1024
    print(f"Total Model Size for {model_path}: {total_size_kb:.2f} KB")
  else:
    print(f"Total Model Size for {model_path}: {total_size} bytes")  # Keep as bytes for small sizes

In [23]:
model_path = "TinyLlama-1.1B-Chat-v1.0-awq"  # Replace with your actual path
calculate_model_size(model_path)

Total Model Size for TinyLlama-1.1B-Chat-v1.0-awq: 732.04 MB


In [ ]:
awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Now you can use the loaded tokenizer and model for inference tasks
# For example, generating text:
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

print(f"Generated Text: {generated_text}")


In [24]:
from src.models.utils_llm import prompt_large_language_model

input_text = "Once upon a time"

# prompt_large_language_model(model, tokenizer, input_text, device)
# prompt_large_language_model(model_bnb_8bit, tokenizer, input_text, device)
# prompt_large_language_model(model_bnb_4bit, tokenizer, input_text, device)
prompt_large_language_model(awq_model, awq_tokenizer, input_text, device)

ModuleNotFoundError: No module named 'src.algorithms.smash_config_mapping'

## 9. Calculating perplexity

In [ ]:
# Evaluate perplexity on each model
from src.evaluations.evaluate_text_generation import evaluate_perplexity

perplexity_8bit = evaluate_perplexity(model_bnb_8bit, train_dataloader, device, send_to_device=False)
perplexity_4bit = evaluate_perplexity(model_bnb_4bit, train_dataloader, device)
perplexity_original = evaluate_perplexity(model, train_dataloader, device)

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

In [ ]:
seq_len = encodings.input_ids.size(1)

In [ ]:
import torch
from tqdm import tqdm

encodings = tokenizer("\n\n".join(data_module.train_dataset["text"]), return_tensors="pt")
max_length = 2048
stride = 512
seq_len = encodings.input_ids.size(1)

nlls = []
prev_end_loc = 0
for begin_loc in tqdm(range(0, seq_len, stride)):
    end_loc = min(begin_loc + max_length, seq_len)
    trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
    input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
    target_ids = input_ids.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

        # loss is calculated using CrossEntropyLoss which averages over valid labels
        # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
        # to the left by 1.
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood)

    prev_end_loc = end_loc
    if end_loc == seq_len:
        break

ppl = torch.exp(torch.stack(nlls).mean())
print(f"Perplexity of model {model.name_or_path}: {ppl:.2f}")

## 9. Text Streamer

In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

## 10. Code Dump

In [ ]:
import accelerate
print(accelerate.__file__)

import transformers
print(transformers.__file__)

from transformers import BitsAndBytesConfig
print(BitsAndBytesConfig.__dict__)

#!pip index versions accelerate
#!pip install accelerate --force-reinstall
!pip show transformers
!pip index versions transformers
!pip show accelerate
!pip index versions accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, AwqConfig
from accelerate.utils import load_and_quantize_model
from accelerate import init_empty_weights

device="cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)
    
# Setting quantization bits for the model
weight_quantization_bits = 8
double_quant = False

bnb_config = BitsAndBytesConfig(
    load_in_8bit=(weight_quantization_bits == 8),
    load_in_4bit=(weight_quantization_bits == 4),
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=["lm_head"],
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
    # bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    # bnb_4bit_quant_type="nf4"
    bnb_4bit_use_double_quant=double_quant,
)

bnb_config.skip_modules = None

smashed_model_bnb = load_and_quantize_model(model, bnb_quantization_config=bnb_config, device_map=device)
print(smashed_model_bnb.__class__.__name__)

# Calibration Dataset needed - WikiText? Something else because data leakage? TODO: Explore
# smashed_model_awq = AutoModelForCausalLM.from_pretrained(
#     temp_dir, quantization_config=awq_config, trust_remote_code=True
# )

awq_config = AwqConfig(
    bits=4,
    fuse_max_seq_len=512,
    do_fuse=False,
)

model.context_length()


In [ ]:
import torch

dataset = [{"text": text} for text in wikitext_dataset]

samples = []
n_run = 0
n_samples=512
awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

for data in dataset:
    if isinstance(data, list):
        line_encoded = data
    else:
        line = data["text"]
        line = line.strip()
        line_encoded = awq_tokenizer.encode(line)
    if len(line_encoded) > 512:
        continue
    sample = torch.tensor([line_encoded])
    if sample.numel() == 0:
        continue
    samples.append(sample)
    n_run += 1
    if n_run == n_samples:
        break
# now concatenate all samples and split according to block size
print(samples)
cat_samples = torch.cat(samples, dim=1)